# Assignment: From Sound to Tokens

**Unit:** Deep Dive: Audio Generation | **Companion reading:** Turning Sound into Numbers; The Sound Dictionary
**Suggested time:** about 1 hour

---

## What is this assignment trying to do?

In the lessons you followed sound from a wave, to a spectrogram, to a short list of tokens from a "sound dictionary". You will build a tiny version of each step yourself:

1. **Part 1:** Turn MIDI note numbers into frequencies, play a melody, and hear what happens when the sample rate is too low. *(TODO 1)*
2. **Part 2:** Cut audio into overlapping frames and turn them into a spectrogram. *(TODO 2)*
3. **Part 3:** Snap spectrogram frames to their nearest codebook entry, then stack codebooks the way residual vector quantization (RVQ) does. *(TODO 3)*

There are only **three short coding tasks** and **three short written reflections**. The reflections are a big part of your grade, because the goal is to understand what you see and hear, not just to make the code run.

### How this assignment is graded

Your functions are **autograded**: the autograder runs them on its **own** small inputs. So:

- Keep every function **name and signature** exactly as given.
- Code outside the three graded functions (plots, audio players) is not run by the autograder.
- Your written reflections are also checked, and may be read by staff.

Fill in every `# TODO`, then run the cells below it to see your code at work.

---
## Setup

A CPU is enough: nothing is downloaded and no GPU is needed. Run the next cell.

In [ ]:
# Colab already has NumPy and Matplotlib, so there is nothing to install.
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

SEED = 0
SR = 16000          # sample rate used in this notebook: 16,000 samples per second
np.random.seed(SEED)

---
## Part 1: From Notes to Sound

In Lesson 2 you saw that MIDI stores a note as a whole number $n$ and that the frequency doubles every 12 steps, anchored at Concert A ($n = 69$, 440 Hz):

$$f(n) = 440 \times 2^{(n - 69)/12}$$

### TODO 1: `midi_to_hz(n)`

Return the frequency in Hz. It will be used on single numbers **and** on NumPy arrays of note numbers.

In [ ]:
def midi_to_hz(n):
    """Frequency in Hz of MIDI note n (a number or a NumPy array)."""
    return None  # TODO 1

### Play a melody (PROVIDED)

The next cell builds each note from a fundamental plus a few weaker harmonics (Lesson 1), then strings the notes together. Check your function first: MIDI 60 should be about 261.63 Hz and MIDI 72 should be twice that.

In [ ]:
def synth_note(freq, duration, sr=SR, harmonics=(1.0, 0.5, 0.25)):
    """One note: a fundamental plus weaker harmonics, with a short fade in and a decay."""
    t = np.arange(int(duration * sr)) / sr
    wave = sum(a * np.sin(2 * np.pi * (k + 1) * freq * t) for k, a in enumerate(harmonics))
    envelope = np.minimum(1.0, t / 0.01) * np.exp(-3.0 * t)
    return 0.3 * wave * envelope / sum(harmonics)


def render_melody(notes, note_length=0.4, sr=SR):
    """Turn a list of MIDI note numbers into one waveform."""
    return np.concatenate([synth_note(midi_to_hz(n), note_length, sr) for n in notes])


print("MIDI 60 ->", round(float(midi_to_hz(60)), 2), "Hz")
print("MIDI 72 ->", round(float(midi_to_hz(72)), 2), "Hz")

MELODY = [60, 60, 67, 67, 69, 69, 67, 65, 65, 64, 64, 62, 62, 60,
          67, 67, 65, 65, 64, 64, 62, 67, 67, 65, 65, 64, 64, 62]
song = render_melody(MELODY)
print(f"{len(MELODY)} notes -> {len(song):,} samples ({len(song) / SR:.1f} s at {SR:,} samples per second)")

t = np.arange(int(0.02 * SR)) / SR
plt.figure(figsize=(8, 2.5))
plt.plot(t * 1000, song[:len(t)])
plt.xlabel("time (ms)"); plt.ylabel("amplitude"); plt.title("The first 20 ms of the melody")
plt.show()
display(Audio(song, rate=SR))

### Sampling too slowly (PROVIDED)

Now we sample one pure high note (MIDI 111, about 4,978 Hz) at two sample rates and ask the computer which frequency it actually recorded. Listen to both.

In [ ]:
def peak_frequency(x, sr):
    """The strongest frequency in x, found with a Fourier transform (Lesson 1)."""
    spectrum = np.abs(np.fft.rfft(x * np.hanning(len(x))))
    return np.fft.rfftfreq(len(x), d=1 / sr)[np.argmax(spectrum)]


high = float(midi_to_hz(111))
for sr in [16000, 8000]:
    tone = synth_note(high, 1.0, sr=sr, harmonics=(1.0,))
    print(f"sample rate {sr:>5} Hz | Nyquist limit {sr // 2:>4} Hz | "
          f"played {high:,.0f} Hz | recorded {peak_frequency(tone, sr):,.0f} Hz")
    display(Audio(tone, rate=sr))

### Part 1 reflection

At 8,000 samples per second, the recorded frequency is not the one that was played. Explain why, and say what would happen to a song if every high note were recorded this way. Answer in a sentence or two.

In [ ]:
part1_reflection = """
TODO: your answer here
"""

---
## Part 2: Building a Spectrogram

A spectrogram (Lesson 1) is made by cutting the waveform into short, overlapping **frames**, running a Fourier transform on each one, and standing the results side by side.

- Each frame is $N$ samples long (the **window**).
- Each new frame starts $H$ samples after the last one (the **hop**).
- Only whole frames are kept, so the number of frames is $1 + \lfloor (\text{len}(x) - N) / H \rfloor$.

### TODO 2: `frame_signal(x, N, H)`

Return a 2-D NumPy array with one frame per **row**: shape `(n_frames, N)`. Row `i` holds `x[i*H : i*H + N]`.

In [ ]:
def frame_signal(x, N, H):
    """Cut the 1-D signal x into overlapping frames of length N, starting every H samples."""
    return None  # TODO 2

### Look at the spectrogram (PROVIDED)

The next cell uses your frames to build a magnitude spectrogram with shape `(N/2 + 1, n_frames)`, frequency by time, just like the worked example in Lesson 1. It then draws the same melody with a short window and a long window.

In [ ]:
def spectrogram(x, N, H):
    """Magnitude spectrogram, shape (N // 2 + 1, n_frames): frequency down the rows, time across."""
    frames = frame_signal(x, N, H) * np.hanning(N)        # taper each frame's edges (Hann window)
    return np.abs(np.fft.rfft(frames, axis=1)).T


def show_spec(S, H, title, ax):
    ax.imshow(np.log(S + 1e-3), origin="lower", aspect="auto", cmap="magma",
              extent=[0, S.shape[1] * H / SR, 0, SR / 2])
    ax.set_ylim(0, 2000); ax.set_xlim(0, 3)
    ax.set_xlabel("time (s)"); ax.set_ylabel("frequency (Hz)"); ax.set_title(title)


for N in [256, 2048]:
    print(f"N = {N:>4}, H = {N // 4:>3}: spectrogram shape {spectrogram(song, N, N // 4).shape}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, N in zip(axes, [256, 2048]):
    show_spec(spectrogram(song, N, N // 4), N // 4, f"window N = {N} ({1000 * N / SR:.0f} ms)", ax)
plt.tight_layout(); plt.show()

### Part 2 reflection

Compare the two pictures (the first 3 seconds are shown). In which one can you see exactly when each note starts? In which one are the harmonics sharp, thin lines? Explain why one window size cannot be best at both. Answer in a sentence or two.

In [ ]:
part2_reflection = """
TODO: your answer here
"""

---
## Part 3: A Tiny Sound Dictionary

A neural codec (Lesson 3) turns each short slice of audio into a vector, then **snaps** that vector to the nearest entry in a codebook and keeps only the entry's ID. Here, each column of a spectrogram plays the role of a frame vector, and the codebooks are learned with a simple clustering method (k-means) that we give you.

### TODO 3: `quantize(z, codebook)`

- `z` has shape `(n, D)`: `n` frame vectors with `D` numbers each.
- `codebook` has shape `(K, D)`: `K` entries.

Return two things:

1. `ids`: a 1-D integer array of length `n`. `ids[i]` is the index of the codebook entry **closest** to `z[i]` (smallest Euclidean distance).
2. `z_hat`: shape `(n, D)`, the chosen entries themselves, so `z_hat[i] = codebook[ids[i]]`.

Hint: NumPy broadcasting gives every distance at once: `z[:, None, :] - codebook[None, :, :]` has shape `(n, K, D)`.

In [ ]:
def quantize(z, codebook):
    """Snap each row of z to its nearest codebook entry. Returns (ids, z_hat)."""
    ids = None     # TODO 3a
    z_hat = None   # TODO 3b
    return ids, z_hat

### One codebook, then several (PROVIDED)

The next cell builds the frame vectors (log spectrogram columns, $N = 512$, $H = 128$), learns codebooks, and uses **your** `quantize` in two ways:

- **VQ:** one codebook of 16 entries.
- **RVQ:** codebook 1 snaps each frame, codebook 2 snaps what is left over (the residual), and so on. The decoder adds the chosen entries together.

In [ ]:
def kmeans(X, K, iters=25, seed=SEED):
    """Learn K codebook entries for the rows of X (simple k-means, uses your quantize)."""
    rng = np.random.default_rng(seed)
    codebook = X[rng.choice(len(X), K, replace=False)].copy()
    for _ in range(iters):
        ids, _ = quantize(X, codebook)
        for k in range(K):
            if np.any(ids == k):
                codebook[k] = X[ids == k].mean(axis=0)
    return codebook


def train_rvq(X, n_books, K):
    """Learn n_books codebooks, each one on the residual left by the ones before it."""
    books, residual = [], X.copy()
    for _ in range(n_books):
        book = kmeans(residual, K)
        _, snapped = quantize(residual, book)
        residual = residual - snapped
        books.append(book)
    return books


def rvq_encode_decode(X, books):
    """Encode X with the stacked codebooks. Returns the token IDs (n, n_books) and the rebuilt X."""
    residual, rebuilt, all_ids = X.copy(), np.zeros_like(X), []
    for book in books:
        ids, snapped = quantize(residual, book)
        rebuilt = rebuilt + snapped
        residual = residual - snapped
        all_ids.append(ids)
    return np.stack(all_ids, axis=1), rebuilt


N_FFT, HOP, K = 512, 128, 16
frames = np.log(spectrogram(song, N_FFT, HOP) + 1e-3).T      # shape (n_frames, 257): one vector per frame
print("frame vectors:", frames.shape)

books = train_rvq(frames, n_books=4, K=K)
errors, rebuilt_by_k = [], {}
for k in range(1, 5):
    ids, rebuilt = rvq_encode_decode(frames, books[:k])
    errors.append(np.mean((frames - rebuilt) ** 2))
    rebuilt_by_k[k] = rebuilt
print("token IDs for the first 5 frames (one column per codebook):")
print(ids[:5])

plt.figure(figsize=(6, 3))
plt.bar([f"{k} codebook" + ("s" if k > 1 else "") for k in range(1, 5)], errors, color="#2563eb")
plt.ylabel("reconstruction error"); plt.title("Error left after each codebook")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, (title, X) in zip(axes, [("original", frames), ("1 codebook", rebuilt_by_k[1]), ("4 codebooks", rebuilt_by_k[4])]):
    ax.imshow(X.T, origin="lower", aspect="auto", cmap="magma",
              extent=[0, len(X) * HOP / SR, 0, SR / 2])
    ax.set_ylim(0, 2000); ax.set_xlim(0, 3); ax.set_title(title); ax.set_xlabel("time (s)")
axes[0].set_ylabel("frequency (Hz)")
plt.tight_layout(); plt.show()

frames_per_second = SR / HOP
print(f"raw audio:      {SR:,} numbers per second")
print(f"4 codebooks:    {frames_per_second:.0f} frames/s x 4 = {4 * frames_per_second:.0f} tokens per second, "
      f"each one of only {K} choices")

### Part 3 reflection

Look at the error chart and the three spectrograms. What did the first codebook get right, and what did the extra codebooks add? Then compare the tokens per second with the raw samples per second, and explain why this matters for a Transformer that generates audio one token at a time. Answer in a sentence or two.

In [ ]:
part3_reflection = """
TODO: your answer here
"""

---
## Submitting

1. Make sure every `TODO` is complete, and all three reflections (`part1_reflection` to `part3_reflection`) are written.
2. Run all cells from top to bottom (**Runtime → Run all** in Colab).
3. Download the notebook as a Python file (**File → Download → Download .py**) and rename it `firstname_lastname_audio.py`.
4. Submit that `.py` file on Gradescope (see the assignment page on the course website for the link).

Your audio and plots are not submitted: the autograder checks your functions and your written answers.